# Dynamo Sandbox

Interactive notebook for inspecting data loading, fragment sampling, and (eventually) model inputs/outputs.

**Kernel**: Use the `vae-env-cluster` environment.

In [ ]:
import sys
from pathlib import Path

# Ensure repo root is importable
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".git").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

BUILD_DIR = REPO_ROOT / "morphseq_playground" / "metadata" / "build06_output"
print(f"Repo root: {REPO_ROOT}")
print(f"Build dir: {BUILD_DIR}")
print(f"Available: {BUILD_DIR.exists()}")

## 1. Load Trajectories

Load one or more experiments and project `z_mu_b` columns into PC space.

In [ ]:
from dev.dynamo.data.loading import load_trajectories

EXPERIMENT_IDS = ["20251207_pbx"]
N_COMPONENTS = 10

ds = load_trajectories(
    experiment_ids=EXPERIMENT_IDS,
    build_dir=BUILD_DIR,
    n_components=N_COMPONENTS,
    verbose=True,
)

In [ ]:
import numpy as np

print(f"Trajectories: {len(ds)}")
print(f"PC dimensions: {ds.n_components}")
print(f"Classes: {ds.class_names}")
print(f"Class → idx: {ds.class_to_idx}")
print()

# Per-class summary
from collections import Counter
class_counts = Counter(t.perturbation_class for t in ds.trajectories)
lengths = [len(t.trajectory) for t in ds.trajectories]
for cls, count in sorted(class_counts.items()):
    cls_lengths = [len(t.trajectory) for t in ds.trajectories if t.perturbation_class == cls]
    print(f"  {cls}: {count} embryos, {np.median(cls_lengths):.0f} median frames")

print(f"\nTrajectory lengths: min={min(lengths)}, median={np.median(lengths):.0f}, max={max(lengths)}")
print(f"PCA variance explained: {ds.pca.explained_variance_ratio_.sum()*100:.1f}%")

## 2. Visualize Trajectories in PC Space

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Color palette per class
COLORS = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A", "#19D3F3"]
class_colors = {cls: COLORS[i % len(COLORS)] for i, cls in enumerate(ds.class_names)}

fig = go.Figure()
for traj in ds.trajectories:
    z = traj.trajectory
    fig.add_trace(go.Scatter3d(
        x=z[:, 0], y=z[:, 1], z=z[:, 2],
        mode="lines",
        line=dict(width=2, color=class_colors[traj.perturbation_class]),
        name=traj.perturbation_class,
        legendgroup=traj.perturbation_class,
        showlegend=False,
        hovertext=traj.embryo_id,
    ))

# One legend entry per class
for cls, color in class_colors.items():
    fig.add_trace(go.Scatter3d(
        x=[None], y=[None], z=[None],
        mode="markers", marker=dict(size=6, color=color),
        name=cls, legendgroup=cls, showlegend=True,
    ))

var = ds.pca.explained_variance_ratio_ * 100
fig.update_layout(
    title="Embryo Trajectories in PC Space",
    scene=dict(
        xaxis_title=f"PC1 ({var[0]:.1f}%)",
        yaxis_title=f"PC2 ({var[1]:.1f}%)",
        zaxis_title=f"PC3 ({var[2]:.1f}%)",
    ),
    width=800, height=600,
)
fig.show()

In [ ]:
# 2D trajectory view: PC1 vs PC2, colored by developmental time
import matplotlib.pyplot as plt
import matplotlib.cm as cm

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: colored by genotype
ax = axes[0]
for traj in ds.trajectories:
    z = traj.trajectory
    ax.plot(z[:, 0], z[:, 1], alpha=0.4, linewidth=0.8,
            color=class_colors[traj.perturbation_class])
# Legend
for cls, color in class_colors.items():
    ax.plot([], [], color=color, label=cls, linewidth=2)
ax.legend(fontsize=8)
ax.set_xlabel(f"PC1 ({var[0]:.1f}%)")
ax.set_ylabel(f"PC2 ({var[1]:.1f}%)")
ax.set_title("Trajectories by genotype")

# Right: colored by time (first embryo highlighted)
ax = axes[1]
for traj in ds.trajectories:
    ax.plot(traj.trajectory[:, 0], traj.trajectory[:, 1],
            alpha=0.1, linewidth=0.5, color="gray")

# Highlight 3 representative embryos with time coloring
rng = np.random.default_rng(42)
sample_idxs = rng.choice(len(ds.trajectories), size=min(3, len(ds.trajectories)), replace=False)
for idx in sample_idxs:
    traj = ds.trajectories[idx]
    z = traj.trajectory
    t = (traj.time_seconds - traj.time_seconds[0]) / 3600  # hours
    sc = ax.scatter(z[:, 0], z[:, 1], c=t, cmap="viridis", s=8, zorder=5)
    ax.plot(z[:, 0], z[:, 1], alpha=0.5, linewidth=1, color="black", zorder=4)

plt.colorbar(sc, ax=ax, label="Time (hours from start)")
ax.set_xlabel(f"PC1 ({var[0]:.1f}%)")
ax.set_ylabel(f"PC2 ({var[1]:.1f}%)")
ax.set_title("Time progression (3 highlighted embryos)")

plt.tight_layout()
plt.show()

## 3. PCA Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scree plot
ax = axes[0]
var_exp = ds.pca.explained_variance_ratio_ * 100
cum_var = np.cumsum(var_exp)
ax.bar(range(1, len(var_exp)+1), var_exp, alpha=0.7, label="Individual")
ax.plot(range(1, len(var_exp)+1), cum_var, "ro-", markersize=5, label="Cumulative")
ax.set_xlabel("PC component")
ax.set_ylabel("Variance explained (%)")
ax.set_title("PCA Scree Plot")
ax.legend()
ax.set_xticks(range(1, len(var_exp)+1))

# Trajectory length distribution
ax = axes[1]
lengths = [len(t.trajectory) for t in ds.trajectories]
ax.hist(lengths, bins=30, edgecolor="black", alpha=0.7)
ax.axvline(np.median(lengths), color="red", linestyle="--", label=f"Median={np.median(lengths):.0f}")
ax.set_xlabel("Trajectory length (frames)")
ax.set_ylabel("Count")
ax.set_title("Trajectory Length Distribution")
ax.legend()

plt.tight_layout()
plt.show()

## 4. Fragment Sampling Demo

Create a `FragmentDataset` and draw sample batches.

In [ ]:
from dev.dynamo.data.dataset import FragmentDataset, fragment_collate_fn, worker_init_fn
from torch.utils.data import DataLoader

fds = FragmentDataset(
    ds,
    min_context=3,
    max_context=20,
    horizons=(1, 2, 3, 4),
    epoch_length=256,
)

loader = DataLoader(
    fds,
    batch_size=16,
    collate_fn=fragment_collate_fn,
    num_workers=0,
)

print(f"Dataset length (epoch): {len(fds)}")
print(f"Valid trajectories: {len(fds.valid_indices)} / {len(ds.trajectories)}")
print(f"Batches per epoch: {len(fds) // 16}")

In [ ]:
# Draw one batch and inspect
batch = next(iter(loader))

print("Batch contents:")
print(f"  context:      {batch.context.shape}  (B, L_max, D)")
print(f"  context_mask: {batch.context_mask.shape}  (B, L_max)")
print(f"  target:       {batch.target.shape}  (B, D)")
print(f"  time_deltas:  {batch.time_deltas.shape}  (B, L_max-1)")
print(f"  horizon_dt:   {batch.horizon_dt.shape}  (B,)")
print(f"  delta_t:      {batch.delta_t.shape}  (B,)")
print(f"  temperature:  {batch.temperature.shape}  (B,)")
print(f"  class_idx:    {batch.class_idx.shape}  (B,)")
print(f"  embryo_idx:   {batch.embryo_idx.shape}  (B,)")
print()

# Context lengths in this batch
ctx_lengths = batch.context_mask.sum(dim=1)
print(f"Context lengths: {ctx_lengths.tolist()}")
print(f"Horizon Δt (hours): {(batch.horizon_dt / 3600).tolist()}")
print(f"Classes: {batch.class_idx.tolist()} → {[ds.class_names[i] for i in batch.class_idx.tolist()]}")

## 5. Visualize Sampled Fragments

Show context fragments (solid) + target points (star) overlaid on the full trajectory cloud.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Background: all trajectories in light gray
all_z = np.concatenate([t.trajectory for t in ds.trajectories], axis=0)

for i, ax in enumerate(axes.flat):
    if i >= batch.context.shape[0]:
        ax.set_visible(False)
        continue

    # Background cloud
    ax.scatter(all_z[:, 0], all_z[:, 1], s=0.5, alpha=0.05, color="gray")

    # Context fragment
    L = int(ctx_lengths[i])
    ctx = batch.context[i, :L].numpy()
    tgt = batch.target[i].numpy()
    cls = ds.class_names[batch.class_idx[i]]

    ax.plot(ctx[:, 0], ctx[:, 1], "o-", color="#1f77b4", markersize=4, linewidth=1.5,
            label=f"context (L={L})")
    ax.plot(ctx[0, 0], ctx[0, 1], "s", color="green", markersize=8, zorder=5, label="start")
    ax.plot(tgt[0], tgt[1], "*", color="red", markersize=14, zorder=5, label="target")

    # Dashed line from last context to target
    ax.plot([ctx[-1, 0], tgt[0]], [ctx[-1, 1], tgt[1]], "--", color="red", alpha=0.5)

    h_hrs = batch.horizon_dt[i].item() / 3600
    ax.set_title(f"{cls}\nL={L}, horizon={h_hrs:.1f}h", fontsize=9)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    if i == 0:
        ax.legend(fontsize=7, loc="upper right")

plt.suptitle("Sampled fragments: context (blue) → target (red star)", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Batch Statistics Over Many Samples

Aggregate statistics over a full epoch to verify sampling behavior.

In [ ]:
# Collect stats over all batches in an epoch
all_ctx_lengths = []
all_horizon_dts = []
all_classes = []
all_time_deltas = []

for batch in loader:
    lengths = batch.context_mask.sum(dim=1)
    all_ctx_lengths.extend(lengths.tolist())
    all_horizon_dts.extend(batch.horizon_dt.tolist())
    all_classes.extend(batch.class_idx.tolist())
    # Collect real (non-padded) time deltas
    for i in range(batch.context.shape[0]):
        L = int(lengths[i])
        if L > 1:
            all_time_deltas.extend(batch.time_deltas[i, :L-1].tolist())

all_ctx_lengths = np.array(all_ctx_lengths)
all_horizon_dts = np.array(all_horizon_dts)
all_classes = np.array(all_classes)

print(f"Total samples: {len(all_ctx_lengths)}")
print(f"Context length — mean: {all_ctx_lengths.mean():.1f}, min: {all_ctx_lengths.min()}, max: {all_ctx_lengths.max()}")
print(f"Horizon Δt — mean: {np.mean(all_horizon_dts)/3600:.2f}h, min: {np.min(all_horizon_dts)/3600:.2f}h, max: {np.max(all_horizon_dts)/3600:.2f}h")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Context length distribution
ax = axes[0, 0]
ax.hist(all_ctx_lengths, bins=range(int(all_ctx_lengths.min()), int(all_ctx_lengths.max())+2),
        edgecolor="black", alpha=0.7)
ax.set_xlabel("Context length (frames)")
ax.set_ylabel("Count")
ax.set_title("Context Length Distribution")

# Horizon distribution (in units of experiment Δt)
ax = axes[0, 1]
horizon_hours = all_horizon_dts / 3600
ax.hist(horizon_hours, bins=30, edgecolor="black", alpha=0.7)
ax.set_xlabel("Horizon (hours)")
ax.set_ylabel("Count")
ax.set_title("Prediction Horizon Distribution")

# Class sampling frequency
ax = axes[1, 0]
class_freq = np.bincount(all_classes, minlength=len(ds.class_names))
bars = ax.bar(range(len(ds.class_names)), class_freq, tick_label=ds.class_names,
              color=[COLORS[i % len(COLORS)] for i in range(len(ds.class_names))])
ax.set_ylabel("Count")
ax.set_title("Class Sampling Frequency")
ax.tick_params(axis='x', rotation=30)

# Inter-frame Δt distribution
ax = axes[1, 1]
dt_minutes = np.array(all_time_deltas) / 60
ax.hist(dt_minutes, bins=50, edgecolor="black", alpha=0.7)
ax.axvline(np.median(dt_minutes), color="red", linestyle="--",
           label=f"Median={np.median(dt_minutes):.1f} min")
ax.set_xlabel("Inter-frame Δt (minutes)")
ax.set_ylabel("Count")
ax.set_title("Time Delta Distribution")
ax.legend()

plt.suptitle(f"Sampling Statistics ({len(all_ctx_lengths)} samples)", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Multi-Experiment Loading

Demonstrate loading multiple experiments to verify cross-experiment Δt handling.

In [ ]:
ds_multi = load_trajectories(
    experiment_ids=["20251207_pbx", "20250305"],
    build_dir=BUILD_DIR,
    n_components=10,
    verbose=True,
)

# Compare Δt across experiments
from collections import defaultdict
dt_by_exp = defaultdict(list)
for t in ds_multi.trajectories:
    dt_by_exp[t.experiment_id].append(t.delta_t)

for exp, dts in dt_by_exp.items():
    print(f"  {exp}: Δt = {dts[0]/60:.1f} min, {len(dts)} embryos")

## 8. DataLoader Integration Test

Verify that full DataLoader iteration works end-to-end with proper batching.

In [ ]:
import torch

fds_full = FragmentDataset(ds, min_context=3, max_context=15, horizons=(1, 2, 3, 4), epoch_length=128)

loader_full = DataLoader(
    fds_full,
    batch_size=32,
    collate_fn=fragment_collate_fn,
    num_workers=0,
)

# Iterate full epoch
n_batches = 0
n_samples = 0
for batch in loader_full:
    n_batches += 1
    n_samples += batch.context.shape[0]
    # Verify invariants
    assert batch.context.shape[2] == N_COMPONENTS
    assert batch.target.shape[1] == N_COMPONENTS
    assert (batch.horizon_dt > 0).all()
    assert batch.context_mask.any(dim=1).all()  # every sample has at least 1 real frame

print(f"Iterated {n_batches} batches, {n_samples} total samples")
print("All invariants passed.")

---

## 9. Model Input/Output Visualization (Placeholder)

Sections below will be populated as model components are built.

### 9a. Potential Landscape (φ₀)
Visualize the learned baseline potential on a 2D slice of PC space.

### 9b. Drift Field
Quiver plot of the learned drift f(z) on a grid.

### 9c. Forward Predictions
Compare context → predicted fan → actual target for individual embryos.

### 9d. Mode Loadings (cₑ)
Scatter plot of embryo-level mode loadings colored by perturbation class.

### 9e. Three-Model Comparison
Side-by-side NLL: kernel baseline vs φ₀-only vs full model.

In [ ]:
# Placeholder: will be filled when models are implemented
print("Model components not yet implemented — see build sequence in CLAUDE.md")